# SQD Post processing

In [ ]:
import os
import json
import time
import pickle
import logging

import numpy as np
import matplotlib.pyplot as plt

from functools import partial
from pathlib import Path

import pyscf
from pyscf import gto, scf, mp, mcscf, cc, ci, tools, ao2mo
from pyscf.scf import chkfile
from pyscf import lib
from pyscf.tools import fcidump
from functools import reduce

from qiskit_ibm_runtime import RuntimeDecoder

from qiskit_addon_sqd.fermion import (
    SCIResult,
    diagonalize_fermionic_hamiltonian,
    solve_sci_batch,
)

# ──────────────────────────────────────────────────────────────────────────────
# Logging
# ──────────────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

## Initial parameters

In [ ]:
shift_array=['-0.15','0.0','0.15','0.3','0.45','0.6','0.75','0.9','1.05','1.2']
shift='0.0'#

charge=3
spin_number=4
medium='solvent'

num_orbitals=17
num_elec_a,num_elec_b=13,9
num_qubits=num_orbitals*2

backend_name='ibm_pittsburgh'

CLASSICAL_DATA_DIR=f'../../classical_methods/{medium}/charge{charge}_spin{spin_number}'
RESULTS_DIR=f'../../sampler_runs/{medium}/charge{charge}_spin{spin_number}/{num_qubits}qubits'
as_label = 'norb_%d_nelec_%d_%d'%(num_orbitals,num_elec_a,num_elec_b)

SQD_RESULTS_DIR= f'charge{charge}_spin{spin_number}/{num_qubits}qubits/shift_{shift}'
Path(SQD_RESULTS_DIR).mkdir(parents=True,exist_ok=True)

## Load results of classical calculations

In [ ]:
def load_converged_scf(CLASSICAL_DATA_DIR,shift,medium):
    """Load a converged ROHF calculation from a checkpoint file."""

    shift_sign = 'minus_' if shift[0]=='-' else ''
    shift_fname_core = shift_sign + shift.strip('-').split('.')[0] + '_' + shift.split('.')[1][::-1].zfill(2)[::-1]
    chkfile_path=f'{CLASSICAL_DATA_DIR}/chk_files/shift_{shift_fname_core}.chk'
    mol = lib.chkfile.load_mol(Path(chkfile_path))
    mf = pyscf.scf.ROHF(mol).density_fit()
    mo_avas=None
    if medium=='gas':
        mo_coeff=np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy')
        mf.__dict__.update(chkfile.load(chkfile_path, 'scf'))
        mf.converged=True
        mf.mo_coeff=mo_coeff
    if medium=='solvent':
        mo_coeff=np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy')
        mf = mf.PCM()
        mf.with_solvent.method = 'IEF-PCM'
        mf.with_solvent.eps = 78.3553
        mf.__dict__.update(chkfile.load(chkfile_path, 'scf'))
        mf.converged=True
        mf.mo_coeff=mo_coeff
        mc = mcscf.CASCI(mf, ncas=num_orbitals, nelecas=(num_elec_a, num_elec_b))
        mc = mc.PCM()
        mc.with_solvent.method = mf.with_solvent.method
        mc.with_solvent.eps = mf.with_solvent.eps
        mc.mo_coeff=mo_coeff
        mo_avas=mo_coeff
        return mf,mc,mo_avas
    return mf

def load_converged_scf_and_integrals(CLASSICAL_DATA_DIR,shift,medium):
    """Load a converged ROHF calculation from a checkpoint file. Does not require fcidump."""

    shift_sign = 'minus_' if shift[0]=='-' else ''
    shift_fname_core = shift_sign + shift.strip('-').split('.')[0] + '_' + shift.split('.')[1][::-1].zfill(2)[::-1]
    chkfile_path=f'{CLASSICAL_DATA_DIR}/chk_files/shift_{shift_fname_core}.chk'
    mol = lib.chkfile.load_mol(Path(chkfile_path))
    mf = pyscf.scf.ROHF(mol).density_fit()
    mo_avas=[]
    h1e,h2e,ecore=[],[],[]
    mycas=[]
    if medium=='gas':
        mo_coeff=np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy') ## after AVAS
        mf.__dict__.update(chkfile.load(chkfile_path, 'scf'))
        mf.converged=True
        mf.mo_coeff=mo_coeff

        mycas = mcscf.CASCI(mf, ncas=num_orbitals, nelecas=(num_elec_a, num_elec_b))
        mycas.mo_coeff = np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy')
        h1e, ecore = mycas.get_h1eff()
        h2e_comp = mycas.get_h2eff()
        h2e = ao2mo.restore(1, h2e_comp, mycas.ncas)

    if medium=='solvent':
        mo_coeff=np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy')
        mf = mf.PCM()
        mf.with_solvent.method = 'IEF-PCM'
        mf.with_solvent.eps = 78.3553
        mf.__dict__.update(chkfile.load(chkfile_path, 'scf'))
        mf.converged=True
        mf.mo_coeff=mo_coeff

        mycas = mcscf.CASCI(mf, ncas=num_orbitals, nelecas=(num_elec_a, num_elec_b)).PCM()
        mycas.with_solvent.method = mf.with_solvent.method
        mycas.with_solvent.eps = mf.with_solvent.eps
        #mycas.with_solvent.frozen = True   
        mycas.mo_coeff = np.load(f'{CLASSICAL_DATA_DIR}/orbitals/shift_{shift}/mo_coeff_{as_label}.npy')
        h1e, ecore = mycas.get_h1eff()       # includes PCM reaction-field operator
        h2e_comp   = mycas.get_h2eff()       # PCM does NOT touch the 2e part
        h2e        = ao2mo.restore(1, h2e_comp, mycas.ncas)
    print(type(mf),type(mycas),type(h1e),type(h2e),type(ecore))
    return mf,mycas,h1e,h2e,ecore


def load_integrals(CLASSICAL_DATA_DIR,shift):
    fname = f'{CLASSICAL_DATA_DIR}/active_spaces/shift_%s/spin_%d_shift_%s_%s.fcidump'%(shift,spin_number,shift,as_label)
    data=fcidump.read(fname)
    h1e=data['H1']
    h2e_comp=data['H2']
    h2e=ao2mo.restore(1,h2e_comp,num_orbitals)
    ecore=data['ECORE']
    return h1e,h2e,ecore

def load_reference_energy(CLASSICAL_DATA_DIR,as_label,idx):
    results = np.load(f'{CLASSICAL_DATA_DIR}/active_space_results_classical_{as_label}.npy',allow_pickle=True).item()
    method='CCSD'
    ccsd_etot=results['%s_%s'%(as_label,method)]
    #print(ccsd_etot)
    return ccsd_etot[idx]

In [ ]:
# mf=load_converged_scf(CLASSICAL_DATA_DIR,shift,medium) ### uses fcidump
# h1e_cas,h2e_cas,ecore=load_integrals(CLASSICAL_DATA_DIR,shift) ### uses fcidump
mf,mc,h1e_cas,h2e_cas,ecore=load_converged_scf_and_integrals(CLASSICAL_DATA_DIR,shift,medium) ### Without fcidump
nuclear_repulsion_energy=ecore
reference_energy= load_reference_energy(CLASSICAL_DATA_DIR,as_label,shift_array.index(shift))#CCSD energy

## Load Hardware Results

In [ ]:
def load_hardware_results(
    results_dir: str,
    num_qubits: int,
    shift: float,
    backend_name: str,
):
    results_file=str(RESULTS_DIR)+f'/qubits_{num_qubits}_shift_{shift}_{backend_name}.json'

    # results_file = os.path.join(
    #     results_dir,
    #     f"qubits_{num_qubits}_shift_{shift}_{backend_name}.json",
    # )
    log.info("Loading hardware results from %s", results_file)

    with open(results_file) as f:
        pub_result = json.load(f, cls=RuntimeDecoder)

    num_shots = pub_result.data.meas.num_shots
    num_bits = pub_result.data.meas.num_bits
    log.info(
        "Loaded BitArray: %d shots × %d bits", num_shots, num_bits
    )

    return pub_result


## SQD

### SQD Parameters

In [ ]:
n_scrf_iter: int = 5
max_iterations: int = 5
energy_tol: float = 1e-5
occupancies_tol: float = 1e-3
num_batches: int = 3
samples_per_batch: int = 200
symmetrize_spin: bool = not bool(spin_number)
carryover_threshold: float = 1e-4
max_cycle: int = 200
spin_sq: float = mf.spin_square()[0]
seed: int = 12345

### SQD Post processing loop

In [ ]:
def update_rdm(casci_object, dmas):
    mo_coeff = casci_object.mo_coeff
    ncore = casci_object.ncore
    ncas = casci_object.ncas

    mocore = mo_coeff[:, :ncore]
    mocas = mo_coeff[:, ncore:ncore + ncas]

    # Core contribution: doubly occupied orbitals
    dm_core = np.dot(mocore, mocore.conj().T) * 2

    # Active-space contribution: back-rotate rdm1 from CAS MO → AO
    dm_cas = reduce(np.dot, (mocas, dmas, mocas.conj().T))

    rho_ao = dm_core + dm_cas
    return rho_ao


In [ ]:
def run_sqd_inner(
    h1e: np.ndarray,
    h2e: np.ndarray,
    bit_array,
    num_orbitals: int,
    num_elec_a: int,
    num_elec_b: int,
    nuclear_repulsion_energy: float,
    # S-CORE options
    max_iterations: int = 10,
    energy_tol: float = 1e-3,
    occupancies_tol: float = 1e-3,
    # Batch / solver options
    num_batches: int = 3,
    samples_per_batch: int = 300,
    symmetrize_spin: bool = True,
    carryover_threshold: float = 1e-4,
    max_cycle: int = 200,
    spin_sq: float = 0.0,
    seed: int = 12345,
):

    # Configure the SCI solver with the target ⟨S²⟩ and max Davidson cycles
    sci_solver = partial(
        solve_sci_batch,
        spin_sq=spin_sq,
        max_cycle=max_cycle,
    )

    # Accumulator for intermediate results from each S-CORE iteration
    result_history: list[list[SCIResult]] = []
    t_start = time.time()

    def callback(results: list[SCIResult]):
        """
        Called at the end of each S-CORE iteration with the batch results.
        Logs energies and subspace dimensions for monitoring convergence.
        """
        result_history.append(results)
        iteration = len(result_history)
        log.info("─── S-CORE iteration %d ───", iteration)
        for i, res in enumerate(results):
            e_total = res.energy + nuclear_repulsion_energy
            dim = np.prod(res.sci_state.amplitudes.shape)
            elapsed = (time.time() - t_start) / 60.0
            log.info(
                "  Batch %d  │  E_total = %.10f Ha  │  dim = %d  │  %.1f min",
                i, e_total, dim, elapsed,
            )

    # ── Main SQD call ──────────────────────────────────────────────────────
    log.info("Starting SQD post-processing (%d orbitals, %d↑ + %d↓ electrons)",
             num_orbitals, num_elec_a, num_elec_b)
    log.info("  max_iterations=%d, num_batches=%d, samples_per_batch=%d",
             max_iterations, num_batches, samples_per_batch)

    result = diagonalize_fermionic_hamiltonian(
        h1e,
        h2e,
        bit_array,
        samples_per_batch=samples_per_batch,
        norb=num_orbitals,
        nelec=(num_elec_a, num_elec_b),
        num_batches=num_batches,
        energy_tol=energy_tol,
        occupancies_tol=occupancies_tol,
        max_iterations=max_iterations,
        sci_solver=sci_solver,
        symmetrize_spin=symmetrize_spin,
        carryover_threshold=carryover_threshold,
        callback=callback,
        seed=seed,
    )

    wall_time = time.time() - t_start
    best_energy = result.energy + nuclear_repulsion_energy
    log.info("SQD complete in %.1f min  │  Best E_total = %.10f Ha",
             wall_time / 60.0, best_energy)

    return result, result_history, wall_time

In [ ]:
def run_sqd_solvent(
    mc,
    bit_array,
    num_orbitals: int,
    num_elec_a: int,
    num_elec_b: int,
    n_scrf_iter: int = 5,
    # S-CORE options
    max_iterations: int = 10,
    energy_tol: float = 1e-3,
    occupancies_tol: float = 1e-3,
    # Batch / solver options
    num_batches: int = 3,
    samples_per_batch: int = 300,
    symmetrize_spin: bool = True,
    carryover_threshold: float = 1e-4,
    max_cycle: int = 200,
    spin_sq: float = 0.0,
    seed: int = 12345,
):
    # ── Validate that the CASCI object has solvent attached ────────────────
    if not hasattr(mc, "with_solvent"):
        raise AttributeError(
            "The CASCI object `mc` does not have a solvent model attached.\n"
            "Ensure you created it with e.g.:\n"
            "    mc = mcscf.CASCI(mf, ncas, nelecas)\n"
            "    mc = solvent.ddCOSMO(mc)  # or mc.PCM()\n"
        )

    # ── Initialise solvent state ──────────────────────────────────────────
    with_solvent_e = None   # scalar solvent polarisation energy
    with_solvent_v = None   # Fock-like potential matrix (nao, nao)

    scrf_data = []
    total_wall = 0.0

    for k in range(n_scrf_iter):
        log.info("=" * 70)
        log.info("SCRF ITERATION %d / %d", k + 1, n_scrf_iter)
        log.info("=" * 70)

        # ── Step 1: update core Hamiltonian ───────────────────────────────
        if with_solvent_v is not None:
            # Subsequent iterations: augment h_core with solvent potential.
            # Use a default-argument closure to capture the current value of
            # with_solvent_v (avoids late-binding issues with lambda).
            _v = with_solvent_v.copy()
            mc.get_hcore = lambda *args, _vv=_v: mc._scf.get_hcore() + _vv
            log.info("  h_core updated: gas-phase + V_solvent")
        else:
            # First iteration: gas-phase core Hamiltonian only.
            mc.get_hcore = lambda *args: mc._scf.get_hcore()
            log.info("  h_core: gas-phase (no solvent potential yet)")

        # ── Step 2: recompute CAS integrals ───────────────────────────────
        h1e_cas, ecore = mc.get_h1eff()
        h2e_cas = ao2mo.restore(1, mc.get_h2eff(), num_orbitals)
        log.info("  ecore = %.10f Ha", ecore)

        # ── Step 3: inner S-CORE + batched diagonalisation ────────────────
        result, inner_history, wt = run_sqd_inner(
            h1e=h1e_cas,
            h2e=h2e_cas,
            bit_array=bit_array,
            num_orbitals=num_orbitals,
            num_elec_a=num_elec_a,
            num_elec_b=num_elec_b,
            nuclear_repulsion_energy=ecore,
            max_iterations=max_iterations,
            energy_tol=energy_tol,
            occupancies_tol=occupancies_tol,
            num_batches=num_batches,
            samples_per_batch=samples_per_batch,
            symmetrize_spin=symmetrize_spin,
            carryover_threshold=carryover_threshold,
            max_cycle=max_cycle,
            spin_sq=spin_sq,
            seed=seed,
        )
        total_wall += wt
        e_active = result.energy
        rdm1_cas = result.rdm1     # spin-summed 1-RDM in CAS MO basis

        # ── Step 4: back-rotate rdm1 to full AO basis ────────────────────
        rho_ao = update_rdm(mc, rdm1_cas).copy()

        # ── Step 5: energy correction ─────────────────────────────────────
        if with_solvent_e is not None:
            # Double-counting correction: the solvent potential V_solvent is
            # already embedded inside h1e_cas via get_h1eff().  The
            # expectation value ⟨V_solvent⟩ = Tr(V_solvent · ρ_AO) has thus
            # been counted once in e_active.  The correct total free energy
            # (Eq. 3) requires only ½ of this term, so we subtract the
            # overcounted half: E = e + ecore + E_solv − Tr(V·ρ).
            e_dup = np.einsum("ij,ji->", with_solvent_v, rho_ao)
            e_total = e_active + ecore + with_solvent_e - e_dup
            log.info("  E_active = %.10f", e_active)
            log.info("  E_solv   = %.10f", with_solvent_e)
            log.info("  E_dup    = %.10f", e_dup)
        else:
            # First iteration: no solvent correction yet.
            e_dup = None
            e_total = e_active + ecore

        log.info("  E_total  = %.10f Ha", e_total)

        # ── Step 6: compute IEF-PCM solvent response ──────────────────────
        with_solvent_e, with_solvent_v = mc.with_solvent._get_vind(rho_ao)
        log.info(
            "  Solvent response computed: E_solv = %.10f", with_solvent_e
        )

        # ── Record this iteration ─────────────────────────────────────────
        scrf_data.append({
            "scrf_iter":      k,
            "e_active":       float(e_active),
            "ecore":          float(ecore),
            "e_total":        float(e_total),
            "e_solvent":      float(with_solvent_e) if with_solvent_e is not None else None,
            "e_double_count": float(e_dup) if e_dup is not None else None,
            "rho_ao":         rho_ao,
            "rdm1_cas":       rdm1_cas,
            "inner_history":  inner_history,
            "wall_time":      wt,
        })

    log.info("=" * 70)
    log.info(
        "SCRF loop complete in %.1f min  |  Final E_total = %.10f Ha",
        total_wall / 60.0, scrf_data[-1]["e_total"],
    )
    log.info("=" * 70)

    return scrf_data, result


## Analysis Functions

In [ ]:
def plot_scrf_convergence(
    scrf_data: list[dict],
    reference_energy: float = None,
    save_path: str = None,
):
    """
    Three-panel diagnostic figure for the SQD IEF-PCM workflow:
        Left   — Total energy vs. SCRF iteration
        Center — Energy error vs. SCRF iteration (log scale, if reference given)
        Right  — Solvent energy contribution vs. SCRF iteration
    """
    iters = [d["scrf_iter"] for d in scrf_data]
    e_totals = [d["e_total"] for d in scrf_data]
    e_solvents = [d["e_solvent"] for d in scrf_data]

    n_panels = 3 if reference_energy is not None else 2
    fig, axs = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5))

    # ── Panel 1: total energy convergence ─────────────────────────────────
    ax = axs[0]
    ax.plot(iters, e_totals, "o-", linewidth=2, color="#1f77b4", label="E_total")
    if reference_energy is not None:
        ax.axhline(
            y=reference_energy, color="grey", linestyle="--",
            linewidth=1.5, label="CASCI IEF-PCM ref",
        )
    ax.set_xlabel("SCRF Iteration", fontsize=12)
    ax.set_ylabel("Total Energy (Ha)", fontsize=12)
    ax.set_title("SQD IEF-PCM Energy Convergence")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(iters)

    # ── Panel 2: energy error (if reference available) ────────────────────
    panel_idx = 1
    if reference_energy is not None:
        ax = axs[panel_idx]
        errors = [abs(e - reference_energy) for e in e_totals]
        ax.plot(iters, errors, "s-", linewidth=2, color="#d62728", label="Error")
        ax.set_yscale("log")
        chem_acc = 1.6e-3
        ax.axhline(
            y=chem_acc, color="#BF5700", linestyle="--",
            linewidth=1.5, label="Chemical accuracy (1.6 mHa)",
        )
        ax.set_xlabel("SCRF Iteration", fontsize=12)
        ax.set_ylabel("|E_SQD - E_ref| (Ha)", fontsize=12)
        ax.set_title("Energy Error vs. SCRF Iteration")
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xticks(iters)
        panel_idx += 1

    # ── Final panel: solvent energy ───────────────────────────────────────
    ax = axs[panel_idx]
    # First iteration may have None for e_solvent; replace with 0 for plotting
    e_solv_plot = [e if e is not None else 0.0 for e in e_solvents]
    ax.plot(iters, e_solv_plot, "^-", linewidth=2, color="#2ca02c", label="E_solvent")
    ax.set_xlabel("SCRF Iteration", fontsize=12)
    ax.set_ylabel("Solvent Energy (Ha)", fontsize=12)
    ax.set_title("IEF-PCM Solvent Energy")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(iters)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        log.info("Figure saved to %s", save_path)
    else:
        plt.show()


def plot_inner_convergence(
    scrf_data: list[dict],
    ecore_per_iter: list[float] = None,
    save_path: str = None,
):
    """
    Plot the S-CORE inner-loop convergence for each SCRF iteration.

    Each SCRF iteration's inner S-CORE history is plotted as a separate
    line showing the best batch energy vs. inner iteration number.
    """
    fig, ax = plt.subplots(1, 1, figsize=(10, 6))

    for d in scrf_data:
        k = d["scrf_iter"]
        ecore = d["ecore"]
        inner_hist = d["inner_history"]

        best_energies = []
        for step_results in inner_hist:
            best_e = min(r.energy + ecore for r in step_results)
            best_energies.append(best_e)

        inner_iters = list(range(len(best_energies)))
        ax.plot(
            inner_iters, best_energies, "o-",
            label=f"SCRF iter {k}", alpha=0.8,
        )

    ax.set_xlabel("S-CORE Inner Iteration", fontsize=12)
    ax.set_ylabel("Best E_total (Ha)", fontsize=12)
    ax.set_title("Inner S-CORE Convergence per SCRF Iteration")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
        log.info("Figure saved to %s", save_path)
    else:
        plt.show()


def print_summary(
    scrf_data: list[dict],
    reference_energy: float = None,
):
    """Print a concise summary table of the SCRF iterations."""
    print("\n" + "=" * 80)
    print("  SQD IEF-PCM POST-PROCESSING SUMMARY")
    print("=" * 80)
    print(f"  Active space       : ({num_elec_a + num_elec_b}e, {num_orbitals}o)")
    print(f"  SCRF iterations    : {len(scrf_data)}")
    print(f"  S-CORE iters/cycle : {max_iterations} (max)")
    print(f"  Batches / S-CORE   : {num_batches}")
    print(f"  Samples per batch  : {samples_per_batch}")

    total_wall = sum(d["wall_time"] for d in scrf_data)
    print(f"  Total wall time    : {total_wall / 60:.1f} min")

    print("-" * 80)
    header = (
        f"  {'SCRF':>4s}  {'E_active (Ha)':>18s}  {'ecore (Ha)':>16s}"
        f"  {'E_total (Ha)':>18s}  {'E_solvent (Ha)':>16s}"
    )
    if reference_energy is not None:
        header += f"  {'Error (mHa)':>12s}"
    print(header)
    print("-" * 80)

    for d in scrf_data:
        e_solv_str = (
            f"{d['e_solvent']:16.8f}" if d["e_solvent"] is not None
            else "        N/A     "
        )
        line = (
            f"  {d['scrf_iter']:4d}"
            f"  {d['e_active']:18.10f}"
            f"  {d['ecore']:16.10f}"
            f"  {d['e_total']:18.10f}"
            f"  {e_solv_str}"
        )
        if reference_energy is not None:
            err_mha = abs(d["e_total"] - reference_energy) * 1000
            line += f"  {err_mha:12.4f}"
        print(line)

    print("-" * 80)
    final_e = scrf_data[-1]["e_total"]
    print(f"  Final E_total      : {final_e:.10f} Ha")

    if reference_energy is not None:
        final_err = abs(final_e - reference_energy)
        print(f"  Reference energy   : {reference_energy:.10f} Ha")
        print(f"  Final error        : {final_err:.6e} Ha"
              f"  ({final_err * 627.509:.4f} kcal/mol)")
        if final_err < 1.6e-3:
            print("  [OK] Within chemical accuracy (1 kcal/mol ~ 1.6 mHa)")
        else:
            print("  [!!] Not at chemical accuracy — consider more SCRF iters"
                  " or larger samples_per_batch")

    # Solvation free energy estimate
    if len(scrf_data) >= 2:
        # Compare final solvated energy to first (gas-phase-like) energy
        delta_g = scrf_data[-1]["e_total"] - scrf_data[0]["e_total"]
        print(f"\n  Approximate solvation correction: {delta_g:.6f} Ha"
              f"  ({delta_g * 627.509:.4f} kcal/mol)")
        print("  (NOTE: for a rigorous G_solv, compare to a separate gas-phase"
              " SQD calculation.)")

    print("=" * 80 + "\n")


## MAIN

In [ ]:
#if __name__ == "__main__":

# ── Validate inputs ───────────────────────────────────────────────────
if mc is None:
    raise ValueError(
        "PySCF CASCI object `mc` must be loaded.\n"
        "Set it in the classical-loading section above. It must have a solvent model attached\n"
        "(mc.with_solvent) and valid mo_coeff from AVAS."
    )

if not hasattr(mc, "with_solvent"):
    raise AttributeError(
        "The CASCI object does not have a solvent model.\n"
        "Create it with e.g.:  mc = solvent.ddCOSMO(mc)  or  mc.PCM()"
    )

# ── Ensure output directory exists ────────────────────────────────────
Path(SQD_RESULTS_DIR).mkdir(parents=True, exist_ok=True)

# ── Load quantum samples ──────────────────────────────────────────────
pub_result = load_hardware_results(
    RESULTS_DIR, num_qubits, shift, backend_name,
)
bit_array = pub_result.data.meas

# ── Run the SCRF + SQD post-processing ────────────────────────────────
scrf_data, final_result = run_sqd_solvent(
    mc=mc,
    bit_array=bit_array,
    num_orbitals=num_orbitals,
    num_elec_a=num_elec_a,
    num_elec_b=num_elec_b,
    n_scrf_iter=n_scrf_iter,
    max_iterations=max_iterations,
    energy_tol=energy_tol,
    occupancies_tol=occupancies_tol,
    num_batches=num_batches,
    samples_per_batch=samples_per_batch,
    symmetrize_spin=symmetrize_spin,
    carryover_threshold=carryover_threshold,
    max_cycle=max_cycle,
    spin_sq=spin_sq,
    seed=seed,
)

# ── Save results ──────────────────────────────────────────────────────
# Save the SCRF data (minus large arrays for portability)
out_pkl = os.path.join(
    SQD_RESULTS_DIR,
    f"sqd_solvent_scrf_{num_qubits}q_{backend_name}.pkl",
)
# Create a serializable version (strip rho_ao to save space if desired)
save_data = []
for d in scrf_data:
    save_record = {k: v for k, v in d.items() if k != "inner_history"}
    # Convert ndarrays to lists for JSON compatibility (optional)
    save_data.append(save_record)
with open(out_pkl, "wb") as f:
    pickle.dump(scrf_data, f)
log.info("Full SCRF result history saved to %s", out_pkl)

# Also save a lightweight JSON summary
out_json = os.path.join(
    SQD_RESULTS_DIR,
    f"sqd_solvent_summary_{num_qubits}q_{backend_name}.json",
)
json_summary = []
for d in scrf_data:
    json_summary.append({
        "scrf_iter":      d["scrf_iter"],
        "e_active":       d["e_active"],
        "ecore":          d["ecore"],
        "e_total":        d["e_total"],
        "e_solvent":      d["e_solvent"],
        "e_double_count": d["e_double_count"],
        "wall_time_s":    d["wall_time"],
    })
with open(out_json, "w") as f:
    json.dump(json_summary, f, indent=2)
log.info("JSON summary saved to %s", out_json)

# ── Print summary ─────────────────────────────────────────────────────
print_summary(scrf_data, reference_energy)

# ── Plot convergence ──────────────────────────────────────────────────
scrf_plot_path = os.path.join(
    SQD_RESULTS_DIR,
    f"sqd_solvent_convergence_{num_qubits}q_{backend_name}.png",
)
plot_scrf_convergence(scrf_data, reference_energy, save_path=scrf_plot_path)

inner_plot_path = os.path.join(
    SQD_RESULTS_DIR,
    f"sqd_solvent_inner_convergence_{num_qubits}q_{backend_name}.png",
)
plot_inner_convergence(scrf_data, save_path=inner_plot_path)

log.info("Done.")
